In [1]:
import os

In [2]:
%pwd

'd:\\Chicken-Disease-Classification-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Chicken-Disease-Classification-Project'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [12]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [14]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [17]:
class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_file(self):
        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

        file_path = self.config.local_data_file

        # 🔁 If file exists → validate it
        if os.path.exists(file_path):
            if zipfile.is_zipfile(file_path):
                logger.info("Valid zip file already exists. Skipping download.")
                return
            else:
                logger.warning("Corrupted file detected. Deleting...")
                os.remove(file_path)

        # ⬇️ Download fresh file
        filename, headers = request.urlretrieve(
            url=self.config.source_URL,
            filename=file_path
        )

        # ✅ Validate after download
        if not zipfile.is_zipfile(filename):
            os.remove(filename)
            raise ValueError("Downloaded file is not a valid zip file.")

        logger.info(f"{filename} downloaded successfully")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        file_path = self.config.local_data_file

        # ✅ Double safety check
        if not zipfile.is_zipfile(file_path):
            raise ValueError("File is not a valid zip file")

        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

        logger.info("Extraction completed successfully")


In [18]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-24 10:21:53,523: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-24 10:21:53,529: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-24 10:21:53,535: INFO: common: created directory at: artifacts_root]
[2026-04-24 10:21:53,539: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-24 10:21:53,546: WARNING: 10769371: Corrupted file detected. Deleting...]
[2026-04-24 10:23:46,059: INFO: 10769371: artifacts/data_ingestion/data.zip downloaded successfully]
[2026-04-24 10:23:47,986: INFO: 10769371: Extraction completed successfully]
